In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DateType,
    TimestampType,
    FloatType,
)
import pyspark.sql.functions as F
catalog_name = 'ecommerce'

In [0]:
df_bronze_brands=spark.read.table(f'{catalog_name}.bronze.brz_brands')
df_bronze_brands.show(10)


In [0]:
# remove the extraspace from the brand_name
df_silver=df_bronze_brands.withColumn('brand_name',F.trim(F.col("brand_name")))
df_silver.show()

In [0]:
# remove extra characters from the brand_name regexp(col,checkcahetr, replace)
df_silver=df_silver.withColumn('brand_code',F.regexp_replace(F.col("brand_code"),r'[^A-Za-z0-9]',''))
df_silver.show()


In [0]:
df_silver.select('category_code').distinct().show()


In [0]:
# we can see different category with same intention like toy toys
anamolyis={
    'GROCERY':"GRCY",
    'BOOKS':'BKS',
    'TOYS':'TOY'
}
df_silver=df_silver.replace(anamolyis,subset=["category_code"])
df_silver.select('category_code').distinct().show()

In [0]:
df_silver.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable(f'{catalog_name}.silver.slv_brands')

### **Category**

In [0]:
df_bronze_category=spark.read.table(f'{catalog_name}.bronze.brz_category')
df_bronze_category.show()

In [0]:
df_silver_category=df_bronze_category.select("category_code").groupBy('category_code').count()
df_silver_category.show()

In [0]:
# upeer case for the categoru_code and drop the duplicates
df_silver_category=df_bronze_category.dropDuplicates(['category_code'])
df_silver_category=df_bronze_category.withColumn('category_code',F.upper(F.col("category_code")))

df_silver_category.show(truncate=False)

In [0]:
df_silver_category.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable(f'{catalog_name}.silver.slv_category')

### **Product**

In [0]:
df_bronze_product=spark.read.table(f'{catalog_name}.bronze.brz_product')
df_bronze_product.show(truncate=False)

row_count,column_count=df_bronze_product.count(),len(df_bronze_product.columns)
print(f'row_count={row_count},column_count={column_count}')


In [0]:
df_bronze_product.select('weight_grams').show(truncate=False)

In [0]:
# removing the g from the weight_gram column and casting it to integer
df_silver_product=df_bronze_product.withColumn('weight_grams',F.regexp_replace('weight_grams','g','').cast(IntegerType()))
df_silver_product.select('weight_grams').show(truncate=False)



In [0]:
# df_silver_product.show(truncate=False)
# remove comma from the length_cm column and casting it to float
df_silver_product=df_silver_product.withColumn('length_cm',F.regexp_replace('length_cm',',','.').cast(FloatType()))
df_silver_product.show(5)
df_silver_product.printSchema()


In [0]:
# convert upeer case category_code and brand_code
df_silver_product=df_silver_product.withColumn('category_code',F.upper(F.col("category_code"))).withColumn('brand_code',F.upper(F.col("brand_code")))
df_silver_product.select("brand_code",'category_code').show(10)

In [0]:
df_silver_product.show(truncate=False)

In [0]:
df_silver_product.select("material").distinct().show()

In [0]:
# correct spelling of material 
df_silver_product=df_silver_product.withColumn("material",F.when(F.col("material")=='Coton','Cotton').when(F.col("material")=='Alumium','Aluminum').when(F.col('material')=="Ruber","Rubber").otherwise(F.col("material"))) # it is for not changing other values in the data frame 
df_silver_product.select("material").show()

In [0]:
# convert rating_count to positive
df_silver_product=df_silver_product.withColumn('rating_count',F.when(F.col('rating_count')<0,F.abs(F.col('rating_count'))).otherwise(F.col("rating_count")))
# a.show()
df_silver_product.filter(F.col('rating_count')<0).select('product_id','rating_count').show()
df_silver_product.show()
                               


In [0]:
# it will be used in future for the table 
df_silver_product=df_silver_product.drop('file_name','ingest_timestamp')
df_silver_product.show(truncate=False)

In [0]:
df_silver_product.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable(f'{catalog_name}.silver.slv_product')

### **customer**

In [0]:
df_bronze_customer=spark.read.table(f'{catalog_name}.bronze.brz_customer')
df_bronze_customer.show(truncate=False)



In [0]:
# cutomer Id is null and drop 
#df_bronze_customer.filter(F.col('customer_id').isNull()).show()

df_silver_customer=df_bronze_customer.dropna(subset=['customer_id'])
df_silver_customer.filter(F.col('customer_id').isNull()).show()

# df_

In [0]:
# phone number null as not avilable 
# df_silver_customer.filter(F.col('phone').isNull()).show()

df_silver_customer=df_silver_customer.withColumn('phone',F.when(F.col('phone').isNull(),'not available').otherwise(F.col('phone')))
df_silver_customer.filter(F.col('phone').isNull()).show()

In [0]:
df_silver_customer.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable(f'{catalog_name}.silver.slv_customer')


In [0]:
df_bronze_calender=spark.read.table(f'{catalog_name}.bronze.brz_calendar')
df_bronze_calender.show(truncate=False)


### **Date**

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calendar")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze.show(3)

In [0]:
df_bronze.printSchema()


In [0]:
from pyspark.sql.functions import to_date


# Convert the string column to a date type
df_silver = df_bronze.withColumn("date", to_date(df_bronze["date"], "dd-MM-yyyy"))

In [0]:
print(df_silver.printSchema())

df_silver.show(5)

In [0]:
    # Find duplicate rows in the DataFrame
duplicates = df_silver.groupBy('date').count().filter("count > 1")

# Show the duplicate rows
print("Total duplicated Rows: ", duplicates.count())
display(duplicates)

In [0]:
# Capitalize first letter of each word in day_name
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))

df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn("week_of_year", F.abs(F.col("week_of_year")))  # Convert negative to positive

df_silver.show(3)

In [0]:
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))

df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))

df_silver.show(3)

In [0]:
# Rename a column
df_silver = df_silver.withColumnRenamed("week_of_year", "week")

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_calendar)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")